# Aim 2 extension: more Tahoe plates and OP3 (planning and download)

Two extensions of the real-data tests:

1. **More Tahoe plates.** The variant-based genotype test found only one drug (adagrasib) on plates 6, 13 and 14. Section 1 reads the metadata of all 14 plates and lists, per plate, the targeted drugs that have a matching variant rule and at least two carrier and two non-carrier cell lines. Section 2 downloads only the plates that add such drugs (each plate is 20 to 30 GB).
2. **OP3.** Section 3 looks for the OP3 data already used in Aim 1 (folder `../Proposal2`), so that it does not have to be downloaded again.

Analysis of the new data goes in a separate notebook (`aim2_extend_analysis.ipynb`) once the files are here. All outputs: `results/cache/aim2_extend/`.

In [1]:
print("f")

f


In [2]:
import os, re, json, time, signal, subprocess, urllib.request
import numpy as np
import pandas as pd

T_START = time.time()
OUT_DIR = "results/cache/aim2_extend"
TAHOE_H5AD_DIR = "data/tahoe100m_h5ad"
TAHOE_META = "data/tahoe100m_metadata/metadata"
WELL_MAP_ALL = "results/cache/aim2_download/tahoe_sample_wells_inferred.csv"   # all 14 plates
BUCKET = "arc-institute-virtual-cell-atlas"
PREFIX = "tahoe100M/2025-02-25/h5ad/"
MIN_CARRIERS = 2
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TAHOE_H5AD_DIR, exist_ok=True)

def stamp(t0, what):
    print(f"[{what}] {time.time() - t0:.1f}s | total {(time.time() - T_START) / 60:.1f} min")

## 1. Which plates carry targeted drugs?
A cell line counts as a carrier only if it has the variant the drug is designed for (same rules as section 22 of `aim2_model_real_data.ipynb`).

In [3]:
VARIANT_RULES = {
    "KRAS G12C": (["adagrasib", "sotorasib", "divarasib"], {"KRAS"}, r"G12C"),
    "KRAS G12D": (["mrtx1133"], {"KRAS"}, r"G12D"),
    "EGFR activating": (["osimertinib", "almonertinib", "erlotinib", "gefitinib", "afatinib", "dacomitinib",
                         "lazertinib", "furmonertinib", "icotinib", "aumolertinib", "befotertinib"],
                        {"EGFR"}, r"L858R|T790M|G719|L861Q|E746|L747|del|ins"),
    "BRAF V600": (["vemurafenib", "dabrafenib", "encorafenib"], {"BRAF"}, r"V600"),
    "MAPK active": (["trametinib", "cobimetinib", "binimetinib", "selumetinib", "mirdametinib"],
                    {"BRAF", "KRAS", "NRAS", "HRAS"}, r"V600|G12|G13|Q61"),
    "PIK3CA hotspot": (["alpelisib", "inavolisib"], {"PIK3CA"}, r"H1047|E545|E542"),
}

t0 = time.time()
wm = pd.read_csv(WELL_MAP_ALL)
CL = pd.read_parquet(os.path.join(TAHOE_META, "cell_line_metadata.parquet"))
lines = sorted(CL.cell_name.astype(str).unique())
PB6 = "results/cache/aim2_tahoe_shared_well_effect/pseudobulk_plate6.h5ad"
if os.path.exists(PB6):                       # count carriers only among the cell lines in the plates
    import anndata as ad
    lines = sorted(set(ad.read_h5ad(PB6, backed="r").obs["line"].astype(str)) & set(lines))
print("Cell lines used for carrier counts:", len(lines))
var = {l: list(zip(g.Driver_Gene_Symbol.astype(str).str.upper(), g.Driver_ProtEffect_or_CdnaEffect.astype(str)))
       for l, g in CL.groupby(CL.cell_name.astype(str))}

def rule_for(drug):
    d = str(drug).lower()
    for name, rule in VARIANT_RULES.items():
        if any(x in d for x in rule[0]):
            return name, rule
    return None, None

def n_carriers(rule):
    _, genes, rx = rule
    return sum(any(g in genes and re.search(rx, e) for g, e in var.get(l, [])) for l in lines)

rows = []
for (plate, drug), g in wm.groupby(["plate", "drug"]):
    name, rule = rule_for(drug)
    if rule is None:
        continue
    n = n_carriers(rule)
    rows.append(dict(plate=plate, drug=drug, variant=name, n_wells=g.well.nunique() if "well" in g else len(g),
                     n_carriers=n, usable=(n >= MIN_CARRIERS) and (len(lines) - n >= MIN_CARRIERS)))
PLAN = pd.DataFrame(rows).sort_values(["plate", "drug"])
PLAN.to_csv(os.path.join(OUT_DIR, "targeted_drugs_per_plate.csv"), index=False)
print(PLAN.to_string(index=False))
have = sorted({f.split("_")[0] for f in os.listdir(TAHOE_H5AD_DIR) if f.endswith(".h5ad")})
need = sorted(set(PLAN[PLAN.usable].plate) - set(have), key=lambda p: int(re.sub(r"\D", "", p)))
print("\nPlates already here:", have)
print("Plates that add usable targeted drugs:", need)
print("Usable drugs per variant:", PLAN[PLAN.usable].groupby("variant").drug.nunique().to_dict())
stamp(t0, "plan")

Cell lines used for carrier counts: 50
  plate                         drug         variant  n_wells  n_carriers  usable
 plate1                    Adagrasib       KRAS G12C        1           6    True
 plate1                     Afatinib EGFR activating        1           0   False
 plate1                    Alpelisib  PIK3CA hotspot        1           5    True
 plate1                  Binimetinib     MAPK active        1          37    True
 plate1                   Dabrafenib       BRAF V600        1           7    True
 plate1                  Encorafenib       BRAF V600        1           7    True
 plate1                   Trametinib     MAPK active        1          37    True
 plate1                  Vemurafenib       BRAF V600        1           7    True
plate10                    Adagrasib       KRAS G12C        1           6    True
plate10                  Binimetinib     MAPK active        1          37    True
plate10                    Gefitinib EGFR activating       

## 2. Download the needed plates
Plates 1 to 3 hold almost all usable targeted drugs (three BRAF inhibitors, two MEK inhibitors, alpelisib and adagrasib); plates 4 to 12 only repeat adagrasib and the MEK inhibitors. `PRIORITY` therefore starts with plates 3, 1 and 2 (about 61 GB). Set `PRIORITY = need` to download every plate that adds a usable drug.

The download runs in the background (`wget -c`, resumable), so the notebook stays free and closing the kernel does not stop it. Re-running skips finished files. Use the progress cell to follow it, and the stop cell to stop it.

In [4]:
t0 = time.time()
PRIORITY = ["plate4", "plate5", "plate7", "plate8", "plate9"]
#PRIORITY = ["plate10", "plate11", "plate12"]
#PRIORITY = ["plate3", "plate1", "plate2"]    # set to `need` for all plates that add a usable drug
url = f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o?prefix={PREFIX}&maxResults=1000"
items = json.load(urllib.request.urlopen(url)).get("items", [])
files = {it["name"].split("/")[-1]: int(it["size"]) for it in items if it["name"].endswith(".h5ad")}
todo = sorted([f for f in files if f.split("_")[0] in PRIORITY], key=lambda f: PRIORITY.index(f.split("_")[0]))
print("Files to download (in this order):")
for f in todo:
    print(f"  {f}  {files[f] / 1e9:.1f} GB")
print(f"Total: {sum(files[f] for f in todo) / 1e9:.1f} GB")
SCRIPT = os.path.join(OUT_DIR, "download_tahoe.sh")
with open(SCRIPT, "w") as fh:
    fh.write("#!/bin/bash\n")
    for f in todo:
        fh.write(f"wget -c -q -P {TAHOE_H5AD_DIR} https://storage.googleapis.com/{BUCKET}/{PREFIX}{f}\n")
    fh.write("echo ALL_DONE\n")
print("Script:", SCRIPT)
stamp(t0, "download list")

Files to download (in this order):
  plate4_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad  23.3 GB
  plate5_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad  19.8 GB
  plate7_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad  16.5 GB
  plate8_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad  30.4 GB
  plate9_filt_Vevo_Tahoe100M_WServicesFrom_ParseGigalab.h5ad  18.8 GB
Total: 108.8 GB
Script: results/cache/aim2_extend/download_tahoe.sh
[download list] 0.2s | total 0.4 min


In [5]:
# start in the background
log = open(os.path.join(OUT_DIR, "download_tahoe.log"), "a")
proc = subprocess.Popen(["bash", SCRIPT], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
with open(os.path.join(OUT_DIR, "download_tahoe.pid"), "w") as fh:
    fh.write(str(proc.pid))
print("Started in the background, PID", proc.pid)

Started in the background, PID 389760


In [20]:
# progress: size on disk vs expected
for f in todo:
    p = os.path.join(TAHOE_H5AD_DIR, f)
    got = os.path.getsize(p) if os.path.exists(p) else 0
    print(f"{f.split('_')[0]:>8}: {got / 1e9:5.1f} / {files[f] / 1e9:5.1f} GB ({got / files[f]:.0%})")
!tail -n 2 {OUT_DIR}/download_tahoe.log

  plate4:  23.3 /  23.3 GB (100%)
  plate5:  19.8 /  19.8 GB (100%)
  plate7:  16.5 /  16.5 GB (100%)
  plate8:  30.4 /  30.4 GB (100%)
  plate9:  18.8 /  18.8 GB (100%)
ALL_DONE
ALL_DONE


In [12]:
# stop the download: set STOP = True and run this cell (works after a kernel restart too)
STOP = False
pid_file = os.path.join(OUT_DIR, "download_tahoe.pid")
if STOP:
    try:
        pid = int(open(pid_file).read())
        os.killpg(os.getpgid(pid), signal.SIGTERM)
        print("Stopped PID", pid)
    except (FileNotFoundError, ProcessLookupError, ValueError):
        print("No running download found.")
else:
    print("Not stopping (set STOP = True to stop the download).")

Not stopping (set STOP = True to stop the download).


## 3. Find the OP3 data from Aim 1
Looks in `../Proposal2` for OP3 files (pseudobulk caches or the original h5ad). Send the output, so that the analysis notebook can read them in place.

In [13]:
!find ../Proposal2 -maxdepth 5 \( -iname "*op3*" -o -iname "*GSE279945*" -o -iname "*openproblem*" -o -iname "*neurips*" -o -iname "*sc_counts*" \) 2>/dev/null | head -50
!ls ../Proposal2/data 2>/dev/null
!ls ../Proposal2/cache 2>/dev/null | head -50

../Proposal2/op3_analysis.ipynb
../Proposal2/data/op3
../Proposal2/data/op3/sc_counts_reannotated_with_counts.h5ad
../Proposal2/.ipynb_checkpoints/Download_op3-checkpoint.ipynb
../Proposal2/.ipynb_checkpoints/op3_analysis-checkpoint.ipynb
../Proposal2/.ipynb_checkpoints/op3_benchmark_audit-checkpoint.ipynb
../Proposal2/.ipynb_checkpoints/op3_positive_controls-checkpoint.ipynb
../Proposal2/results/op3_benchmark_audit
../Proposal2/results/op3_audit
../Proposal2/figures/FigS3_op3_positive_controls.png
../Proposal2/figures/FigS3_op3_positive_controls.pdf
../Proposal2/figures/.ipynb_checkpoints/FigS4_op3_positive_controls-checkpoint.pdf
../Proposal2/figures/.ipynb_checkpoints/FigS3_op3_positive_controls-checkpoint.pdf
../Proposal2/figures/.ipynb_checkpoints/FigOP3_positive_controls-checkpoint.png
../Proposal2/figures/FigS4_op3_positive_controls.pdf
../Proposal2/figures/FigS4_op3_positive_controls.png
../Proposal2/figures/FigOP3_positive_controls.png
../Proposal2/Download_op3.ipynb
../Propos

In [14]:
import anndata as ad
op3 = ad.read_h5ad("../Proposal2/data/op3/sc_counts_reannotated_with_counts.h5ad", backed="r")
print(op3)
o = op3.obs
for c in o.columns:
    v = o[c]
    print(f"{c:30s} {str(v.dtype):10s} n_unique={v.nunique():6d}  e.g. {list(v.astype(str).unique()[:5])}")
x = op3.X[:1000]
x = x.toarray() if hasattr(x, "toarray") else x
print("X max:", x.max(), "| integer:", bool((x == x.round()).all()))
print("layers:", list(op3.layers.keys()))
op3.file.close()

AnnData object with n_obs × n_vars = 298087 × 21265 backed at '../Proposal2/data/op3/sc_counts_reannotated_with_counts.h5ad'
    obs: 'dose_uM', 'timepoint_hr', 'well', 'row', 'col', 'plate_name', 'cell_id', 'cell_type', 'split', 'donor_id', 'sm_name', 'control', 'SMILES', 'sm_lincs_id', 'library_id', 'leiden_res1', 'group', 'cell_type_orig', 'plate_well_celltype_reannotated', 'cell_count_by_well_celltype', 'cell_count_by_plate_well'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'celltypist_celltype_colors', 'donor_id_colors', 'hvg', 'leiden_res1_colors', 'log1p', 'neighbors', 'over_clustering', 'rank_genes_groups'
    obsm: 'HTO_clr', 'X_pca', 'X_umap', 'protein_counts'
    obsp: 'connectivities', 'distances'
    layers: None (.X)
dose_uM                        float64    n_unique=     3  e.g. ['0.1', '1.0', '14.1']
timepoint_hr                   float64    n_unique=     1  e.g. ['24.0']
well                           category   n_

In [20]:
# copy the OP3 h5ad from Proposal2 into Proposal3 (once)
import os, shutil, time
SRC = "../Proposal2/data/op3/sc_counts_reannotated_with_counts.h5ad"
DST = "data/op3/sc_counts_reannotated_with_counts.h5ad"
os.makedirs(os.path.dirname(DST), exist_ok=True)
t0 = time.time()
if os.path.exists(DST) and os.path.getsize(DST) == os.path.getsize(SRC):
    print("Already copied.")
else:
    shutil.copy2(SRC, DST)
print(f"{DST}: {os.path.getsize(DST) / 1e9:.2f} GB (source {os.path.getsize(SRC) / 1e9:.2f} GB) | {time.time() - t0:.1f}s")

data/op3/sc_counts_reannotated_with_counts.h5ad: 3.52 GB (source 3.52 GB) | 15.3s
